## 🎬 Sistema de recomendación basado en popularidad y géneros

En este notebook se desarrolla un modelo sencillo de recomendación de películas
a partir del dataset MovieLens.

El objetivo es construir un sistema que sea capaz de sugerir películas relevantes
teniendo en cuenta dos aspectos principales:

- La popularidad de las películas (basada en valoraciones de usuarios)
- Los géneros que interesan al usuario

Para ello, se sigue un enfoque progresivo:

1. Se limpian y preparan los datos
2. Se construye un modelo baseline basado en popularidad
3. Se mejora el modelo incorporando filtrado por géneros
4. Se implementa una función final que permite obtener recomendaciones fácilmente

Este enfoque permite pasar de un modelo muy básico a uno más útil y cercano a
un sistema real de recomendación.

## Bloque 1: Importación de CSV

In [15]:
import pandas as pd

# Cargar datos procesados
ratings = pd.read_csv("../../../data/processed/ratings_sample.csv")
movies = pd.read_csv("../../../data/processed/movies_clean.csv")

# Vista inicial
print("Ratings:")
display(ratings.head())

print("\nMovies:")
display(movies.head())

Ratings:


,userId,movieId,rating,timestamp
0,33183,21,5.0,942800860
1,27473,1375,3.0,973042848
2,160083,4888,2.0,1008171230
3,96819,122906,4.5,1688662566
4,79969,6377,4.0,1395785884



Movies:


,movieId,title,genres
0,1,Toy Story (1995),"['Adventure', 'Animation', 'Children', 'Comedy..."
1,2,Jumanji (1995),"['Adventure', 'Children', 'Fantasy']"
2,3,Grumpier Old Men (1995),"['Comedy', 'Romance']"
3,4,Waiting to Exhale (1995),"['Comedy', 'Drama', 'Romance']"
4,5,Father of the Bride Part II (1995),['Comedy']


## Bloque 2: Asegurar genres como lista

In [16]:
import ast

movies["genres"] = movies["genres"].apply(ast.literal_eval)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


## Bloque 3: Entrenamiento baseline

In [17]:
def train_baseline(ratings, movies, min_ratings=20):
    """
    Modelo baseline por popularidad
    """

    ratings_count = ratings.groupby("movieId")["rating"].count()

    valid_movies = ratings_count[ratings_count >= min_ratings].index

    filtered_ratings = ratings[ratings["movieId"].isin(valid_movies)]

    top_movies = (
        filtered_ratings.groupby("movieId")["rating"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
        .merge(movies, on="movieId")
    )

    return top_movies

## Bloque 4: Recomendador por géneros

In [18]:
def recommend_by_genres(df, input_genres, n=10, min_match=0.75):
    """
    Recomendador con filtro por porcentaje de coincidencia de géneros
    """

    df = df.copy()

    # Calcular score
    df["genre_score"] = df["genres"].apply(
        lambda genres: len(set(genres) & set(input_genres))
    )

    # Calcular porcentaje de cobertura
    df["genre_coverage"] = df["genre_score"] / len(input_genres)

    # Filtrar por mínimo porcentaje
    df_filtered = df[df["genre_coverage"] >= min_match]

    if df_filtered.empty:
     print("⚠️ No hay suficientes coincidencias, relajando filtro...")
     df_filtered = df[df["genre_score"] > 0]

    # Ordenar
    return (
        df_filtered
        .sort_values(by=["rating", "genre_coverage"], ascending=False)
        .head(n)[["title", "rating", "genres", "genre_score", "genre_coverage"]]
    )

## Bloque 5: Entrenar modelo

In [19]:
top_movies = train_baseline(ratings, movies)

top_movies.head()

,movieId,rating,title,genres
0,2973,4.425000,Crimes and Misdemeanors (1989),"[Comedy, Crime, Drama]"
1,1237,4.400000,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",[Drama]
2,318,4.385220,"Shawshank Redemption, The (1994)","[Crime, Drama]"
3,858,4.381773,"Godfather, The (1972)","[Crime, Drama]"
4,202439,4.378788,Parasite (2019),"[Comedy, Drama]"


## Bloque 6: Recomendación por generos

In [20]:
recommendations = recommend_by_genres(
    top_movies,
    input_genres=["Action", "Adventure", "Romance"],
    n=10,
)

recommendations

,title,rating,genres,genre_score,genre_coverage
48,"Princess Bride, The (1987)",4.156000,"[Action, Adventure, Comedy, Fantasy, Romance]",3,1.0
78,North by Northwest (1959),4.110169,"[Action, Adventure, Mystery, Romance, Thriller]",3,1.0
747,Romancing the Stone (1984),3.509091,"[Action, Adventure, Comedy, Romance]",3,1.0
793,"Three Musketeers, The (1993)",3.470588,"[Action, Adventure, Comedy, Romance]",3,1.0
809,True Lies (1994),3.453947,"[Action, Adventure, Comedy, Romance, Thriller]",3,1.0
957,Twister (1996),3.283019,"[Action, Adventure, Romance, Thriller]",3,1.0
991,"Jewel of the Nile, The (1985)",3.232143,"[Action, Adventure, Comedy, Romance]",3,1.0
995,Mr. & Mrs. Smith (2005),3.227273,"[Action, Adventure, Comedy, Romance]",3,1.0


## Bloque 7: Funcion para devolver resultados

In [ ]:
# 🔹 Entrenas UNA vez
model = train_baseline(ratings, movies)

# 🔹 Función solo de recomendación
def get_recommendations(model, genres, n=10):
    return recommend_by_genres(model, genres, n)

In [ ]:
get_recommendations(model, ["Fantasy", "Action", "Drama"], 5)

,title,rating,genres,genre_score,genre_coverage
16,Princess Mononoke (Mononoke-hime) (1997),4.250000,"[Action, Adventure, Animation, Drama, Fantasy]",3,1.0
111,"Lord of the Rings: The Return of the King, The...",4.063348,"[Action, Adventure, Drama, Fantasy]",3,1.0
129,Harry Potter and the Deathly Hallows: Part 2 (...,4.040984,"[Action, Adventure, Drama, Fantasy, Mystery, I...",3,1.0
874,King Kong (2005),3.392857,"[Action, Adventure, Drama, Fantasy, Thriller]",3,1.0
936,Thor (2011),3.305085,"[Action, Adventure, Drama, Fantasy, IMAX]",3,1.0


## 📊 Conclusiones

Se ha desarrollado un sistema de recomendación sencillo pero efectivo que combina:

- Un modelo baseline basado en la media de valoraciones por película
- Un filtrado por contenido utilizando los géneros de las películas

El modelo funciona en dos fases:

1. Entrenamiento:
   Se calcula un ranking global de películas en función de su valoración media,
   filtrando aquellas con pocos votos para evitar resultados poco fiables.

2. Recomendación:
   Se seleccionan únicamente las películas que coinciden con los géneros indicados
   por el usuario, calculando una puntuación de similitud (`genre_score`) y un
   porcentaje de coincidencia (`genre_coverage`).

Además, se implementa un mecanismo de fallback que evita devolver resultados vacíos
cuando no hay coincidencias suficientes, relajando el criterio de filtrado.

Aunque se trata de un modelo sencillo, este sistema ya reproduce ideas clave de
los sistemas de recomendación reales:

- Separación entre entrenamiento e inferencia
- Uso de métricas de relevancia
- Filtrado basado en contenido

Como mejora futura, se podría incorporar información del usuario (historial de
valoraciones) para construir un sistema de recomendación personalizado.